In [ ]:
# start coding here

# %load_ext autoreload
# %autoreload 2

from single_cellm.jointemb.dataset.perturbation_kaggle import KaggleDEGDataModule
from single_cellm.jointemb.single_cellm_lightning import TranscriptomeTextDualEncoderLightning
from single_cellm.jointemb.model import  TranscriptomeTextDualEncoderModel
from single_cellm.jointemb.config import  TranscriptomeTextDualEncoderConfig
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

import torch
import numpy as np
import torch.nn as nn
import wandb
import pandas as pd
from lightning import LightningModule, Trainer
from torch import optim
import shutil 
from transformers import AutoTokenizer
from lightning.pytorch.loggers import Logger, WandbLogger


torch.set_float32_matmul_precision('high')

In [ ]:
# A simple output module that converts the embeddings computed from jointemb into DEG values
# TODO might additionally provide a one-hot of the drug and the cell type
# TODO also might provide the control expression of the cell type in question


def mrrmse_pd(y_pred: pd.DataFrame, y_true: pd.DataFrame):
    
    return ((y_pred - y_true)**2).mean(axis=1).apply(np.sqrt).mean()

def mrrmse_np(y_pred, y_true):
    
    return np.sqrt(np.square(y_true - y_pred).mean(axis=1)).mean()

def mrrmse_torch(y_pred, y_true):
    return torch.sqrt(torch.square(y_true - y_pred).mean(axis=1)).mean()

class JointEmbeddingDEG(LightningModule):
    def __init__(self, output_dim: int = 18211):
        super().__init__()
        self.lightning_model = TranscriptomeTextDualEncoderLightning.load_from_checkpoint(snakemake.input.model)
        self.base_model = self.lightning_model.model
        self.lightning_model.freeze()
        embed_dim = self.base_model.config.projection_dim
        
        self.linear = nn.Linear(embed_dim, output_dim)
        # TODO might add some more layers

    def forward(self, control_features, perturbation_features):
        """
        Mimic the interface of the base model
        """
        _, embedding_control = self.base_model.get_text_features(**control_features)
        _, embedding_perturbation = self.base_model.get_text_features(**perturbation_features)

        # TODO normalize embeddingss or not?
        
        distance = (embedding_perturbation - embedding_control)
        # TODO normalize difference or not?
        # TODO is difference (-) the correct metric?
        
        return self.linear(distance)

    def process_step(self, batch, batch_idx, step_type):
        model_res = self.forward(batch["control_features"], batch["perturbation_features"])
        labels = batch["labels"]
        
        loss = mrrmse_torch(model_res, labels)
        self.log(
            f"{step_type}_loss",
            loss,
            on_step=True,
            on_epoch=True,
            prog_bar=True,
            logger=True,
        )
        
        return loss
    
    def training_step(self, batch, batch_idx):
        return self.process_step(batch, batch_idx, "train")

    def validation_step(self, batch, batch_idx):
        print("in val")
        return self.process_step(batch, batch_idx, "val")
    
    def predict_step(self, batch, batch_idx):
        model_res = self.forward(batch["control_features"], batch["perturbation_features"])
        return model_res
        
    def configure_optimizers(self):
        optimizer = optim.Adam(self.parameters(), lr=1e-3)
        return optimizer

In [ ]:
# load datasets
df = pd.read_parquet(snakemake.input.de)

test_map = pd.read_csv(snakemake.input.test_map)
# add Lincs ID
lincs_ids = df.set_index("sm_name")["sm_lincs_id"].drop_duplicates()
lincs_ids["Dimethyl Sulfoxide"] = "LSM-36361" #  (not necessary actually)
test_map["sm_lincs_id"] = lincs_ids.loc[test_map.sm_name].values

In [ ]:
def train_fold(k):
    model = JointEmbeddingDEG()
    datamodule = KaggleDEGDataModule(df, test_map, snakemake.params.embedding_sentence, snakemake.params.num_folds, batch_size=8)

    early_stop = EarlyStopping(
        monitor="val_loss", min_delta=1e-5, patience=1, verbose=False, mode="min"
    )
    checkpoint_callback = ModelCheckpoint(
        monitor="val_loss",
        mode="min",
        save_top_k=2,
        save_last=True,
        filename="{epoch}-{val_loss:.2f}",
    )

    trainer = Trainer(callbacks=[checkpoint_callback, early_stop], 
                      check_val_every_n_epoch=1,
                      logger=WandbLogger(
                          # save_dir=get_path(["paths", "wandb_logs"]),
                          project="PerturbationKaggle",
                          entity="single-cellm",
                          name=f"fold_{k}",
                          log_model=False,
                          reinit=True,
                          settings=wandb.Settings(start_method='thread'),
                          mode="online" if snakemake.params.wandb_logging else "disabled"
                      )
                     )

    trainer.fit(model=model, datamodule=datamodule)
    
    test_set_results = trainer.predict(datamodule=datamodule)
    wandb.finish()
    res_df = pd.DataFrame(torch.concat(test_set_results).detach().cpu().numpy(), columns=df.columns[5:])
    
    best_val_loss = checkpoint_callback.best_model_score.item()
    #del model, datamodule, early_stop, checkpoint_callback, trainer, test_set_results
    return best_val_loss, res_df

In [ ]:
score, test_set_results  = train_fold(0)


In [ ]:
# prints currently alive Tensors and Variables
import torch
torch.cuda.empty_cache()
import gc
for obj in gc.get_objects():
    try:
        if torch.is_tensor(obj) or (hasattr(obj, 'data') and torch.is_tensor(obj.data)):
            print(type(obj), obj.size())
    except:
        pass

In [ ]:
shutil.copy(checkpoint_callback.best_model_path, snakemake.output["model"])

In [ ]:
# TODO might unfreeze the model and continue

In [ ]:
  # column order seems to correspond to final submission sheet
test_set_results.head()

In [ ]:
test_set_results.index.name = "id"
test_set_results.to_csv(snakemake.output.predictions, index=True)

In [ ]:
# TODO don't forget to use ensembls!